# Exploitation Zone — Knowledge Graph ABOX construction

**Projecte 2 — BDA-GIA (UPC)**

Aquest notebook crea instancies al KG a partir de les taules que surten de la Trusted Zone i la metadata (TBOX.owl), utilitzant  RDFLib Graph.


## 0. Instal·lació de dependències

In [1]:
#!pip install rdflib duckdb pandas

## 1. Imports i configuració

El notebook comienza importando las librerías necesarias para ejecutar el proceso. Las principales son DuckDB, para conectarse a la base de datos de la Trusted Zone, RDFLib, para construir y manipular el grafo RDF y Pandas, para gestionar los datos tabulares durante la transformación.

Además, se importa el módulo urllib.parse, que se utiliza para garantizar que los textos utilizados como fragmentos de URI sean válidos.


También, para que el código sea más legible se definen constantes para las clases y propiedades para evitar escribir el URI completo. 


In [2]:
import duckdb
import pandas as pd
import rdflib
from rdflib import Graph, Namespace, Literal, RDF, RDFS
import urllib.parse
import os
from rdflib import XSD

# ──────────────────────────────────────────────
# CONFIGURACIÓ DE PATHS 

DUCKDB_PATH = "../trust_zone/trusted_zone.db"   # ruta a la Trusted Zone
TBOX_PATH   = "TBOX.owl"                         # TBOX al mateix directori
OUTPUT_KG   = "kg_accidentes_bcn.ttl"            # fitxer de sortida del KG

## 2. Definició de Namespaces

Tots els URIs del TBOX provenen de WebProtégé. Definim el namespace base i les propietats/classes tal com estan al fitxer `TBOX.owl`.

In [3]:
# Namespace base del TBOX (WebProtégé)
WP = Namespace("http://webprotege.stanford.edu/")

# Namespace per a les instàncies (ABOX)
DATA = Namespace("http://data.bcn.cat/accidentes/")

# ── Classes ──────────────────────────────────────────────────────────
C_ACCIDENTE           = WP["RDHGr7pl4B25glJmq2xYYmN"]
C_ACCIDENTE_LEVE      = WP["R7oZUszqj7aEjlsk698JmFM"]
C_ACCIDENTE_GRAVE     = WP["RYqhWnA3Yc3qHHEFBZcRoJ"]
C_ACCIDENTE_MORTAL    = WP["R8bzivJwhQYj45eHewQTg08"]
C_ACCIDENTE_SIN_VIC   = WP["RCyHtlgkOu2iks7ZQbQPeLw"]
C_LUGAR               = WP["RCglr2LgVmFa85WTIfryLqR"]
C_PERSONA             = WP["R8CyWdnPgD3YkoaC4jXyLH8"]
C_CONDUCTOR           = WP["RBPXhVTcni9UaQtHolrwEGM"]
C_PASSATGER           = WP["R9NI0gPxFQhpaSZpsxWyRWC"]
C_VIANANT             = WP["RBjMNvYweizmQ8EE1HnWwNF"]
C_ESTACIO_METEO       = WP["RDfBEkaNQRJf9P95RTve7DV"]
C_LECTURA_METEO       = WP["R8I54xpcV9jUWT2m34cWCxd"]
P_CLOSER_STATION      = WP["R8zRTYFQVvCR9Vg5oCehoMn"]   # Accidente → EstacioMeteo


# ── Object Properties ────────────────────────────────────────────────
P_UBICADO_EN          = WP["Ro80knf6kx29DunELcteYN"]   # Accidente/Estacio → Lugar
P_INVOLUCRADO_EN      = WP["Rqp27L1N9oTZweWd4AbJKw"]   # Persona → Accidente
P_MEDIR               = WP["RDc1x0FCFi3bcbVBIr3mn8a"]   # Estacio → LecturaMeteo

# ── Data Properties ──────────────────────────────────────────────────
P_ACCIDENT_ID         = WP["RiUMyEG1nwNphv16JMNmsZ"]
P_FECHADO_EN          = WP["R8BfMxHvuSS21BCIoJe4qQ8"]
P_NUM_MORT            = WP["RDLF2iVJgjiRFE4TYcqZ6a3"]
P_NUM_FERIT_LLEU      = WP["R8RNUHdxy4iNBMh6DiKMNHR"]
P_NUM_FERIT_GREU      = WP["R9KzZ5w47b2GzTS1wFlEzB6"]
P_CALLE               = WP["R9J0LyEyfULTAx5DDRGG4Ys"]
P_BARRIO              = WP["RCJq0kcAKOAezhCOBeEqGlT"]
P_DISTRITO            = WP["R8VCcKl4uCPbHDX8to2A5w8"]
P_EDAD                = WP["RCsWF1rtGUwHlrCw2nFCiyE"]
P_SEXO                = WP["RDEVg1ttXmeqGODajUtvvgq"]
P_MOTIU_DESPL         = WP["RB1tANRENUq7XbDeRnBbE3J"]
P_VEHICLE_IMPLICAT    = WP["R94iV1JYRs5IqQSGPTpJbM8"]
P_ESTACIO_ID          = WP["R9DzTUzW9I8YOWAWyrQbYXj"]
P_ACRONIMO            = WP["R9REQUrspa6BUM46zxZhfsg"]
P_VALOR               = WP["R7sjUhrXMjtfE85znEWI95G"]

## 3. Lectura de la Trusted Zone (DuckDB)

La Trusted Zone es la capa del flujo de datos en la que las tablas ya han sido formateadas y no presentan problemas de calidad. El notebook se conecta a la base de datos DuckDB en modo de solo lectura y extrae las tres tablas principales mediante una consulta SQL que selecciona toda la tabla.

Las tablas seleccionadas son, la tabla T_ACCIDENTS, que contiene los registros de accidentes con información sobre la ubicación, la fecha, la hora y el número de víctimas, también la tabla T_PERSONAS, que contiene el perfil de las personas implicadas en cada accidente (su tipo, edad, sexo, vehículo y motivo del desplazamiento) y, por último, la tabla T_METEO, que contiene las lecturas meteorológicas de las estaciones de Barcelona. 


Una vez cargados los datos en forma de DataFrames de Pandas, se cierra la conexión con DuckDB y, a partir de este momento, todo el procesamiento se realiza en memoria.

In [4]:
conn = duckdb.connect(DUCKDB_PATH, read_only=True)

df_accidents = conn.execute("SELECT * FROM T_ACCIDENTS").df()
df_persones  = conn.execute("SELECT * FROM T_PERSONES").df()
df_meteo     = conn.execute("SELECT * FROM T_METEO").df()

conn.close()

print(f"T_ACCIDENTS: {len(df_accidents):,} files")
print(f"T_PERSONES:  {len(df_persones):,} files")
print(f"T_METEO:     {len(df_meteo):,} files")
df_accidents.head(3)

T_ACCIDENTS: 7,741 files
T_PERSONES:  16,001 files
T_METEO:     16,396 files


,numero_expedient,codi_districte,nom_districte,codi_barri,nom_barri,codi_carrer,nom_carrer,num_postal,descripcio_dia_setmana,nk_any,...,descripcio_causa_vianant,numero_morts,numero_lesionats_lleus,numero_lesionats_greus,numero_victimes,numero_vehicles_implicats,coordenada_utm_y_ed50,coordenada_utm_x_ed50,longitud_wgs84,latitud_wgs84
0,2025S000175,10,Sant Martí,72,Sant Martí de Provençals,169409,Corts Catalanes,989-991,Diumenge,2025,...,Desconegut,0,0,0,0,1,433080.792,4585086.167,2.198167,41.412694
1,2025S000233,2,Eixample,6,la Sagrada Família,350308,València,Desconegut,Dimarts,2025,...,Desconegut,0,0,0,0,2,431074.797,4583801.691,2.174313,41.400955
2,2025S000273,7,Horta-Guinardó,35,el Guinardó,365702,Mare de Déu de Montserrat,89-103,Dimecres,2025,...,Altres,0,1,0,1,1,430529.797,4585444.552,2.167605,41.415705


In [5]:
df_persones.head(3)

,numero_expedient,codi_districte,nom_districte,codi_barri,nom_barri,codi_carrer,nom_carrer,num_postal,descripcio_dia_setmana,nk__any,...,edat,descripcio_tipus_persona,descripcio_lloc_atropellament_vianant,descripcio_motiu_desplacament_vianant,descripcio_motiu_desplacament_conductor,descripcio_victimitzacio,coordenada_utm_x_ed50,coordenada_utm_y_ed50,longitud_wgs84,latitud_wgs84
0,2025S000172,5,Sarrià-Sant Gervasi,26,Sant Gervasi - Galvany,344101,Gràcia,Desconegut,Dissabte,2025,...,48,Passatger,Desconegut,Desconegut,Desconegut,Ferit lleu: Rebutja assistència sanitària,429099.819,4583362.448,2.150740,41.396827
1,2025S000211,6,Gràcia,28,Vallcarca i els Penitents,158107,Vallcarca,Desconegut,Dilluns,2025,...,55,Conductor,Desconegut,Desconegut,In itínere,Il.lès,428634.486,4585028.124,2.144978,41.411788
2,2025S000228,4,Les Corts,20,la Maternitat i Sant Ramon,701434,Mig (Descendent),K17,Dimarts,2025,...,55,Conductor,Desconegut,Desconegut,Oci i entreteniment,Il.lès,427237.071,4581573.542,2.128675,41.380549


## 4. Construcció del Graf RDF (ABOX)

Creem el graf carregant primer el TBOX i després afegint les instàncies (ABOX) a partir de les taules de DuckDB.

### Funció auxiliar: classificar el tipus d'accident

In [6]:
def accident_class(row):
    """Retorna la classe OWL correcta segons el nombre de víctimes."""
    morts = row["numero_morts"]
    greus = row["numero_lesionats_greus"]
    lleus = row["numero_lesionats_lleus"]
    
    if morts > 0:
        return C_ACCIDENTE_MORTAL
    elif greus > 0:
        return C_ACCIDENTE_GRAVE
    elif lleus > 0:
        return C_ACCIDENTE_LEVE
    else:
        return C_ACCIDENTE_SIN_VIC


def person_class(descripcio):
    """Retorna la subclasse de Persona correcta."""
    if not descripcio:
        return C_PERSONA
    d = str(descripcio).lower()

    if "Conductor" in d:
        return C_CONDUCTOR
    elif "Passatger" in d:
        return C_PASSATGER
    elif "Vianant" in d:
        return C_VIANANT
    return C_PERSONA


def safe_uri(text):
    """Codifica un text per ser usat com a fragment d'URI. Usat per a identificadors de accidents (exp_id), llocs (lloc_key) i codi estacions (codi_est)."""
    return urllib.parse.quote(str(text).strip().replace(" ", "_"), safe="")



### 4.1 Inicialitzar el graf i carregar el TBOX

In [7]:
g = Graph()

# Carregar el TBOX (schema/metamodel)
g.parse(TBOX_PATH)
print(f"TBOX carregat: {len(g)} triples de schema")

# Lligar namespaces per a queries llegibles
g.bind("wp",   WP)
g.bind("data", DATA)
g.bind("rdf",  RDF)
g.bind("rdfs", RDFS)
g.bind("xsd",  XSD)

TBOX carregat: 134 triples de schema


### 4.2 Poblar accidents i llocs

In [8]:
# Conjunt per evitar duplicats de Llocs
llocs_vistos = set()

for _, row in df_accidents.iterrows():
    exp_id = str(row["numero_expedient"]).strip()

    # ── URI de l'accident ──────────────────────────────────────────────
    acc_uri = DATA[f"accident/{safe_uri(exp_id)}"]

    # Tipus (subclasse d'Accidente)
    g.add((acc_uri, RDF.type, accident_class(row)))
    g.add((acc_uri, RDF.type, C_ACCIDENTE))  # classe general també

    # Propietats de l'accident
    g.add((acc_uri, P_ACCIDENT_ID, Literal(exp_id, datatype=XSD.string)))

    if pd.notna(row.get("numero_morts")):
        g.add((acc_uri, P_NUM_MORT, Literal(int(row["numero_morts"]), datatype=XSD.integer)))
    if pd.notna(row.get("numero_lesionats_lleus")):
        g.add((acc_uri, P_NUM_FERIT_LLEU, Literal(int(row["numero_lesionats_lleus"]), datatype=XSD.integer)))
    if pd.notna(row.get("numero_lesionats_greus")):
        g.add((acc_uri, P_NUM_FERIT_GREU, Literal(int(row["numero_lesionats_greus"]), datatype=XSD.integer)))

    # Data i hora com a string ISO (dateTimeStamp requereix timezone)
    any_ = row.get("nk_any", "")
    mes  = row.get("mes_any", "")
    dia  = row.get("dia_mes", "")
    hora = row.get("hora_dia", 0) or 0
    if pd.notna(any_) and pd.notna(mes) and pd.notna(dia):
        dt_str = f"{int(any_):04d}-{int(mes):02d}-{int(dia):02d}T{int(hora):02d}:00:00"
        
        g.add((acc_uri, P_FECHADO_EN, Literal(dt_str, datatype=XSD.dateTimeStamp)))

    # ── URI del Lloc ───────────────────────────────────────────────────
    codi_carrer  = row.get("codi_carrer", "")
    num_postal   = row.get("num_postal", "")
    lloc_key     = f"{codi_carrer}_{num_postal}"
    lloc_uri     = DATA[f"lloc/{safe_uri(lloc_key)}"]

    if lloc_key not in llocs_vistos:
        g.add((lloc_uri, RDF.type, C_LUGAR))
        if pd.notna(row.get("nom_carrer")):
            g.add((lloc_uri, P_CALLE,    Literal(str(row["nom_carrer"]), datatype=XSD.string)))
        if pd.notna(row.get("nom_barri")):
            g.add((lloc_uri, P_BARRIO,   Literal(str(row["nom_barri"]),  datatype=XSD.string)))
        if pd.notna(row.get("nom_districte")):
            g.add((lloc_uri, P_DISTRITO, Literal(str(row["nom_districte"]), datatype=XSD.string)))
        llocs_vistos.add(lloc_key)

    # Relació Accident → Lloc
    g.add((acc_uri, P_UBICADO_EN, lloc_uri))

print(f"✓ Accidents afegits. Total triples ara: {len(g):,}")

✓ Accidents afegits. Total triples ara: 79,606


### 4.3 Poblar persones

In [9]:
for i, row in df_persones.iterrows():
    exp_id   = str(row["numero_expedient"]).strip()
    pers_uri = DATA[f"persona/{safe_uri(exp_id)}_{i}"]
    acc_uri  = DATA[f"accident/{safe_uri(exp_id)}"]

    # Tipus de persona
    g.add((pers_uri, RDF.type, person_class(row.get("descripcio_tipus_persona"))))

    if pd.notna(row.get("edat")):
        g.add((pers_uri, P_EDAD, Literal(int(row["edat"]), datatype=XSD.integer)))
    if pd.notna(row.get("descripcio_sexe")) and str(row["descripcio_sexe"]).strip():
        g.add((pers_uri, P_SEXO, Literal(str(row["descripcio_sexe"]), datatype=XSD.string)))
    if pd.notna(row.get("desc_tipus_vehicle_implicat")) and str(row["desc_tipus_vehicle_implicat"]).strip():
        g.add((pers_uri, P_VEHICLE_IMPLICAT, Literal(str(row["desc_tipus_vehicle_implicat"]), datatype=XSD.string)))

    motiu = row.get("descripcio_motiu_desplacament_conductor") or row.get("descripcio_motiu_desplacament_vianant")
    if pd.notna(motiu) and str(motiu).strip():
        g.add((pers_uri, P_MOTIU_DESPL, Literal(str(motiu), datatype=XSD.string)))

    # Relació Persona → Accident
    g.add((pers_uri, P_INVOLUCRADO_EN, acc_uri))

print(f"✓ Persones afegides. Total triples ara: {len(g):,}")

✓ Persones afegides. Total triples ara: 175,612


### 4.4 Poblar estacions meteorològiques i lectures

In [10]:
estacions_vistes = set()

for i, row in df_meteo.iterrows():
    codi_est  = str(row["codi_estacio"]).strip()
    est_uri   = DATA[f"estacio/{safe_uri(codi_est)}"]

    # Crear estació (una sola vegada)
    if codi_est not in estacions_vistes:
        g.add((est_uri, RDF.type,    C_ESTACIO_METEO))
        g.add((est_uri, P_ESTACIO_ID, Literal(codi_est, datatype=XSD.string)))
        estacions_vistes.add(codi_est)

    # Cada fila és una LecturaMeteorologica
    lect_uri = DATA[f"lectura/{safe_uri(codi_est)}_{i}"]
    g.add((lect_uri, RDF.type,   C_LECTURA_METEO))

    if pd.notna(row.get("acronim")):
        g.add((lect_uri, P_ACRONIMO,   Literal(str(row["acronim"]),  datatype=XSD.string)))
    if pd.notna(row.get("valor")):
        g.add((lect_uri, P_VALOR,      Literal(float(row["valor"]), datatype=XSD.float)))
    if pd.notna(row.get("data_lectura")):
        g.add((lect_uri, P_FECHADO_EN, Literal(row["data_lectura"].isoformat(), datatype=XSD.dateTimeStamp)))

    # Relació Estació → Lectura
    g.add((est_uri, P_MEDIR, lect_uri))

print(f"✓ Dades meteorològiques afegides. Total triples: {len(g):,}")

✓ Dades meteorològiques afegides. Total triples: 257,598


### 4.5 Poblar la relació CloserStation (Accidente → EstacioMeteo)

La propietat `CloserStation` (URI `R8zRTYFQVvCR9Vg5oCehoMn`) connecta cada instància d'`Accidente` amb la seva `EstacioMeteo` més propera. La informació prové de la columna `meteo_estacio_proxima` de la taula `T_UNIFIED` de la base de dades `exploit_zone.db`, que conté el codi de l'estació (coincident amb `codi_estacio` de `T_METEO`).

In [12]:
EXPLOIT_DB_PATH = "../trust_zone/exploit_zone.db"

con_exploit = duckdb.connect(EXPLOIT_DB_PATH, read_only=True)
df_unified = con_exploit.execute(
    "SELECT numero_expedient, meteo_estacio_proxima FROM T_UNIFIED"
).df()
con_exploit.close()

print(f"T_UNIFIED: {len(df_unified):,} files")

closer_count = 0
for _, row in df_unified.iterrows():
    exp_id   = str(row["numero_expedient"]).strip()
    codi_est = str(row["meteo_estacio_proxima"]).strip() if pd.notna(row["meteo_estacio_proxima"]) else None

    if not exp_id or not codi_est:
        continue

    acc_uri = DATA[f"accident/{safe_uri(exp_id)}"]
    est_uri = DATA[f"estacio/{safe_uri(codi_est)}"]

    g.add((acc_uri, P_CLOSER_STATION, est_uri))
    closer_count += 1

print(f"✓ Relacions CloserStation afegides: {closer_count:,}. Total triples ara: {len(g):,}")

T_UNIFIED: 16,001 files
✓ Relacions CloserStation afegides: 16,001. Total triples ara: 265,197


### 4.6 Serialitzar el KG a disc

Una vez que el grafo se ha rellenado con todas las instancias, se serializa en el disco en formato Turtle (extensión .ttl). 


La serialización es el proceso de convertir el grafo en memoria en un archivo de texto estándar que cualquier herramienta compatible con RDF pueda leer posteriormente.

El archivo generado se utilizará tanto en la parte de pattern matching de SPARQL como el notebook que crea embeddings del KG para el aprendizaje automático. Las fases posteriores cargarán este archivo directamente, sin necesidad de volver a leer DuckDB ni reconstruir las tripletas (el grafo).

In [13]:
g.serialize(OUTPUT_KG, format="turtle")
size_mb = os.path.getsize(OUTPUT_KG) / 1_000_000
print(f"✓ KG serialitzat a '{OUTPUT_KG}' ({size_mb:.2f} MB, {len(g):,} triples)")

✓ KG serialitzat a 'kg_accidentes_bcn.ttl' (16.47 MB, 265,197 triples)
